In [1]:
import geopandas as gpd
from rasterstats import zonal_stats
import pandas as pd
import argparse
import os
import numpy as np

In [2]:
print(f"Old Directory: {os.getcwd()}")
os.chdir(r'../')
print(f"New Directory: {os.getcwd()}")

Old Directory: d:\GitHub\Sala_Situacion_Dinagua\Status_Outlook_Bulletin\notebook
New Directory: d:\GitHub\Sala_Situacion_Dinagua\Status_Outlook_Bulletin


In [3]:
'''
# Define arguments 
parser = argparse.ArgumentParser(
                    prog='generate drought warning',
                    description='Postprocess the ESP outlook products',
                    epilog='Jose Valles, DINAGUA, 08Abr2026')

# 
parser.add_argument('ano', help='provide year of analysis')
parser.add_argument('mes', help='provide month of analysis')

args = parser.parse_args()
'''
args = argparse.Namespace(
    ano='2026',
    mes='03',
)

In [4]:
# 1. Cargar las subcuencas
cuencas = gpd.read_file('qgis_status_outlook/shapefiles/Cuencas_n2.shp')
# from cuencas, extract cod_n2, nombrec2, area and create a dataframe
cuencas_df = cuencas[['cod_n2', 'nombrec2', 'area']].copy()
cuencas_df.rename(columns={'cod_n2': 'cod_n2', 'nombrec2': 'nombrec2', 'area': 'area'}, inplace=True)
cuencas_df.head().style.hide(axis='index')

cod_n2,nombrec2,area
10,RÍO CUAREIM,8222
11,RÍO URUGUAY entre Río Cuareim y Río Arapey Grande,2583
12,RÍO ARAPEY CHICO,2152
13,RÍO ARAPEY GRANDE,9698
14,RÍO URUGUAY entre Río Arapey Grande y Río Daymán,1631


In [5]:
# 2. Definir los archivos raster de SPI
# De la carpeta ../qgis_status_outlook/IPE/ extraer los archivos raster de SPI para los periodos 1, 3, 6 y 12 meses. 
# Los nombres de archivo contienen el periodo de tiempo, por ejemplo: YYYY_MM_IPE_01meses, YYYY_MM_IPE_03meses, YYYY_MM_IPE_06meses, YYYY_MM_IPE_12meses. Donde YYYY es el año y MM es el mes.
ano = args.ano
mes = args.mes

raster_files = {
    '1_mes': f'qgis_status_outlook/IPE/{ano}_{mes}_IPE_01meses.tif',
    '3_meses': f'qgis_status_outlook/IPE/{ano}_{mes}_IPE_03meses.tif',
    '6_meses': f'qgis_status_outlook/IPE/{ano}_{mes}_IPE_06meses.tif',
    '12_meses': f'qgis_status_outlook/IPE/{ano}_{mes}_IPE_12meses.tif'
}

In [6]:
# 3. Iterar sobre cada raster y calcular el promedio
for col_name, raster_path in raster_files.items():
    print(f"Procesando: {col_name} - {raster_path}")
    
    # Verificar que el archivo existe
    if not os.path.exists(raster_path):
        print(f"Archivo no encontrado: {raster_path}")
        cuencas[col_name] = None
        continue
    
    # zonal_stats devuelve una lista de diccionarios
    # Asegurar que ambos tienen el mismo CRS y usar all_touched=True para capturar valores
    # Usar band=2 para especificar la banda 2
    stats = zonal_stats(
        cuencas.to_crs(epsg=32721),  # Asegurar CRS común
        raster_path, 
        stats="mean",
        all_touched=True,
        nodata=-999,
        band=1  # Usar la banda 1 del raster
    )
    
    # Extraemos el valor 'mean' y lo asignamos a la columna cod_n2 de el dataframe cuencas_df
    cuencas_df[col_name] = [x['mean'] if x['mean'] is not None else float('nan') for x in stats]
    
    # Mostrar resumen
    valid_values = sum(1 for x in stats if x['mean'] is not None)
    print(f"Valores válidos: {valid_values}/{len(stats)}")

Procesando: 1_mes - qgis_status_outlook/IPE/2026_03_IPE_01meses.tif
Valores válidos: 47/47
Procesando: 3_meses - qgis_status_outlook/IPE/2026_03_IPE_03meses.tif
Valores válidos: 47/47
Procesando: 6_meses - qgis_status_outlook/IPE/2026_03_IPE_06meses.tif
Valores válidos: 47/47
Procesando: 12_meses - qgis_status_outlook/IPE/2026_03_IPE_12meses.tif
Valores válidos: 47/47


In [7]:
# import 01_status_month.csv, 03_status_month.csv
status_1_mes = pd.read_csv('qgis_status_outlook/csvtables/01_status_month.csv')
status_3_meses = pd.read_csv('qgis_status_outlook/csvtables/03_status_month.csv')


In [8]:
# 4. asignar el status_1_mes y status_3_meses a cuencas_df usando cod_n2 como clave
cuencas_df = cuencas_df.merge(
    status_1_mes.rename(columns={'stationID': 'cod_n2', 'category': 'IEH_1'}),
    on='cod_n2', how='left'
)
cuencas_df = cuencas_df.merge(
    status_3_meses.rename(columns={'stationID': 'cod_n2', 'category': 'IEH_3'}),
    on='cod_n2', how='left'
)


In [9]:
def clasificar_ipe(valor):
    if np.isnan(valor):
        return np.nan
    if valor >= -0.5:
        return 0
    elif valor >= -1.0:
        return 1
    elif valor >= -1.5:
        return 2
    elif valor >= -2.0:
        return 3
    else:
        return 4

cuencas_df['IPE_1'] = cuencas_df['1_mes'].apply(clasificar_ipe)
cuencas_df['IPE_3'] = cuencas_df['3_meses'].apply(clasificar_ipe)
cuencas_df['IPE_6'] = cuencas_df['6_meses'].apply(clasificar_ipe)
cuencas_df['IPE_12'] = cuencas_df['12_meses'].apply(clasificar_ipe)


# drop columns 1_mes, 3_meses, 6_meses, 12_meses
# cuencas_df.drop(columns=['1_mes', '3_meses', '6_meses', '12_meses'], inplace=True)



In [10]:
display(cuencas_df.head(10).style.hide(axis='index'))

cod_n2,nombrec2,area,1_mes,3_meses,6_meses,12_meses,IEH_1,IEH_3,IPE_1,IPE_3,IPE_6,IPE_12
10,RÍO CUAREIM,8222,-0.842076,-0.541071,-0.660047,0.231737,3,3,1,1,1,0
11,RÍO URUGUAY entre Río Cuareim y Río Arapey Grande,2583,-0.667619,-0.557101,-0.582893,0.025773,3,3,1,1,1,0
12,RÍO ARAPEY CHICO,2152,-0.640402,-0.548251,-0.594086,0.034191,3,3,1,1,1,0
13,RÍO ARAPEY GRANDE,9698,-0.358728,-0.535235,-0.591636,-0.149829,3,3,0,1,1,0
14,RÍO URUGUAY entre Río Arapey Grande y Río Daymán,1631,-0.359018,-0.570645,-0.413013,-0.255738,3,3,0,1,0,0
15,RÍO DAYMÁN,3415,-0.230650,-0.583759,-0.500265,-0.387460,3,3,0,1,1,0
16,RÍO URUGUAY entre Río Daymán y Río Queguay Grande,1714,-0.130082,-0.619983,-0.477339,-0.481378,3,3,0,1,0,0
17,RÍO QUEGUAY GRANDE,8549,-0.049864,-0.605479,-0.664144,-0.568165,3,2,0,1,1,1
18,RÍO URUGUAY entre Río Queguay Grande y Río Negro,3729,-0.048657,-0.729673,-0.855560,-0.698029,1,2,0,1,1,1
19,RÍO URUGUAY entre Río Negro y Río de la Plata,3636,0.250929,-0.662610,-1.092027,-0.721909,2,2,0,1,2,1


In [11]:
# 6. Evaluar criterios de alerta de sequía por fila

# Normalidad: IEH_1 > 2, IEH_3 >= 3, IPE_3 <= 2, IPE_6 <= 2 → cumple 3 de 4 → 1
cuencas_df['NORMALIDAD'] = (
    (cuencas_df['IEH_1'] > 2).astype(int) +
    (cuencas_df['IEH_3'] >= 3).astype(int) +
    (cuencas_df['IPE_3'] <= 2).astype(int) +
    (cuencas_df['IPE_6'] <= 2).astype(int)
>= 3).astype(int)


In [12]:
cuencas_df['MODERADO'] = ((
    (cuencas_df['IEH_1'] <= 2).astype(int) +
    (cuencas_df['IEH_3'] <= 2).astype(int) +
    (cuencas_df['IPE_3'] >= 3).astype(int) +
    (cuencas_df['IPE_6'] >= 2).astype(int)
>= 2).astype(int) * 2)

In [13]:
cuencas_df['SEVERO'] = ((
    (cuencas_df['IEH_1'] <= 1).astype(int) +
    (cuencas_df['IEH_3'] <= 2).astype(int) +
    (cuencas_df['IPE_6'] >= 3).astype(int) +
    (cuencas_df['IPE_12'] >= 2).astype(int)
>= 3).astype(int) * 3)

In [14]:
cuencas_df['EXTREMO'] = ((
    (cuencas_df['IEH_1'] == 1).astype(int) +
    (cuencas_df['IEH_3'] == 1).astype(int) +
    (cuencas_df['IPE_6'] >= 4).astype(int) +
    (cuencas_df['IPE_12'] >= 3).astype(int)
>= 4).astype(int) * 4)

In [15]:
# 7. Asignar code_advertencia como el valor máximo entre NORMALIDAD, MODERADO, SEVERO, EXTREMO
cuencas_df['code_advertencia'] = cuencas_df[['NORMALIDAD', 'MODERADO', 'SEVERO', 'EXTREMO']].max(axis=1)

In [16]:
# do not show index column
display(cuencas_df[['cod_n2', 'nombrec2', 'code_advertencia']].style.hide(axis='index'))

cod_n2,nombrec2,code_advertencia
10,RÍO CUAREIM,1
11,RÍO URUGUAY entre Río Cuareim y Río Arapey Grande,1
12,RÍO ARAPEY CHICO,1
13,RÍO ARAPEY GRANDE,1
14,RÍO URUGUAY entre Río Arapey Grande y Río Daymán,1
15,RÍO DAYMÁN,1
16,RÍO URUGUAY entre Río Daymán y Río Queguay Grande,1
17,RÍO QUEGUAY GRANDE,1
18,RÍO URUGUAY entre Río Queguay Grande y Río Negro,2
19,RÍO URUGUAY entre Río Negro y Río de la Plata,2


In [17]:
# 8. Crear columna ESTADO_ACTUAL y colorear según code_advertencia
estado_map = {1: 'NORMALIDAD', 2: 'MODERADO', 3: 'SEVERO', 4: 'EXTREMO'}
color_map = {'NORMALIDAD': '#D9D9D9', 'MODERADO': '#FAD463', 'SEVERO': '#F5B183', 'EXTREMO': '#EE4565'}

cuencas_df['ESTADO_ACTUAL'] = cuencas_df['code_advertencia'].map(estado_map).fillna('')

def colorear_estado(val):
    color = color_map.get(val, '')
    return f'background-color: {color}' if color else ''

In [18]:
display(cuencas_df[['cod_n2', 'nombrec2', 'ESTADO_ACTUAL']].
        style.map(colorear_estado, subset=['ESTADO_ACTUAL']).
        hide(axis='index'))

cod_n2,nombrec2,ESTADO_ACTUAL
10,RÍO CUAREIM,NORMALIDAD
11,RÍO URUGUAY entre Río Cuareim y Río Arapey Grande,NORMALIDAD
12,RÍO ARAPEY CHICO,NORMALIDAD
13,RÍO ARAPEY GRANDE,NORMALIDAD
14,RÍO URUGUAY entre Río Arapey Grande y Río Daymán,NORMALIDAD
15,RÍO DAYMÁN,NORMALIDAD
16,RÍO URUGUAY entre Río Daymán y Río Queguay Grande,NORMALIDAD
17,RÍO QUEGUAY GRANDE,NORMALIDAD
18,RÍO URUGUAY entre Río Queguay Grande y Río Negro,MODERADO
19,RÍO URUGUAY entre Río Negro y Río de la Plata,MODERADO


In [19]:
# replace advertenci column from cuencas using the column ESTADO_ACTUAL from cuencas_df
cuencas = cuencas.merge(cuencas_df[['cod_n2', 'ESTADO_ACTUAL']], on='cod_n2', how='left')
# drop advertenci column from cuencas
cuencas.drop(columns=['advertenci'], inplace=True)
# rename ESTADO_ACTUAL to warning
cuencas.rename(columns={'ESTADO_ACTUAL': 'ADVER'}, inplace=True)

In [18]:
# Save cuencas to shapefile. Use in the file name the ano-mes of the data used to calculate the advertencia
cuencas.to_file(f'../qgis_status_outlook/shapefiles/Advertencia_{ano}-{mes}.shp')